# AIND Ephys Dispatch

Build an `AINDEPhysDispatchScanConfig`, expand it with `GridScanGenerationTask`, and run the `AINDEPhysDispatchTask` for each coordinate. The task itself clones [aind-ephys-job-dispatch](https://github.com/AllenNeuralDynamics/aind-ephys-job-dispatch) on first use, invokes its `code/run_capsule.py` with the flags this single config produces, and copies the resulting `job_*.json` into the single config's `coordinate_output_root` (one folder per coordinate, under `obi-output/01_aind_ephys_dispatch/grid_scan/<idx>/`).

Reads the toy SpikeInterface recording from `00_generate_toy_recording.ipynb` (saved at `./output/00_toy_example_recording`).

## Imports

In [1]:
import json
from pathlib import Path

import obi_one as obi
from obi_one.scientific.tasks.aind_ephys._01_dispatch.blocks import (
    DispatchBasic,
    DispatchDataDependent,
    DispatchDebug,
    SpikeInterfaceInfo,
)

## Build the scan config

In [2]:
recording_folder = (
    Path.cwd() / "../../../obi-output/00_toy_example_recording"
).resolve()
assert recording_folder.exists(), (
    f"{recording_folder} not found. Run 00_generate_toy_recording.ipynb first."
)

scan_config = obi.AINDEPhysDispatchScanConfig(
    dispatch_basic=DispatchBasic(
        split_segments=True,
        split_groups=True,
        skip_timestamps_check=False,
        min_recording_duration=1.0,
    ),
    dispatch_data_dependent=DispatchDataDependent(
        input_format="spikeinterface",
        multi_session_data=False,
        spikeinterface_info=SpikeInterfaceInfo(
            reader_type="binaryfolder",
            reader_kwargs={"folder_path": str(recording_folder)},
        ),
    ),
    dispatch_debug=DispatchDebug(
        debug_mode=False,
        debug_duration=10.0,
    ),
)

## Generate the grid scan and run the dispatch task for each coordinate

Per-coordinate output goes to `obi-output/01_aind_ephys_dispatch/grid_scan/<idx>/` (the `ZERO_INDEX` directory option keeps the path stable for downstream notebooks).

In [3]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/01_aind_ephys_dispatch/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)

[2026-04-29 19:02:24,259] INFO: python -u code/run_capsule.py --input spikeinterface --min-recording-duration 1.0 --spikeinterface-info '{"reader_type": "binaryfolder", "reader_kwargs": {"folder_path": "/Users/james/Documents/obi/code/obi-main/obi-output/00_toy_example_recording"}}'


[2026-04-29 19:02:24,531] INFO: Running job dispatcher with the following parameters:
	SPLIT SEGMENTS: True
	SPLIT GROUPS: True
	DEBUG: False
	DEBUG DURATION: 30.0
	SKIP TIMESTAMPS CHECK: False
	MULTI SESSION: False
	INPUT: spikeinterface
	MIN_RECORDING_DURATION: 1.0
	SPIKEINTERFACE_INFO: {"reader_type": "binaryfolder", "reader_kwargs": {"folder_path": "/Users/james/Documents/obi/code/obi-main/obi-output/00_toy_example_recording"}}
Parsing spikeinterface input folder
	Stream name: None
Recording to be processed in parallel:
Relative to: ../data
	block0_None_recording1
		Duration: 10.0 s - Num. channels: 70
Generated 1 job config files



[PosixPath('../../../obi-output/01_aind_ephys_dispatch/grid_scan/0')]

## Inspect the generated job JSONs

We read the per-coordinate outputs straight from each `coordinate_output_root` — no copying.

In [4]:
for single_config in grid_scan.single_configs:
    coord_dir = Path(single_config.coordinate_output_root)
    job_files = sorted(coord_dir.glob("job_*.json"))
    print("---")
    print("idx:                    ", single_config.idx)
    print("min_recording_duration: ", single_config.dispatch_basic.min_recording_duration)
    print("debug_mode:             ", single_config.dispatch_debug.debug_mode)
    print("coordinate_output_root: ", coord_dir)
    print("job files:              ", [p.name for p in job_files])

first = grid_scan.single_configs[0]
first_job = Path(first.coordinate_output_root) / "job_0.json"
if first_job.exists():
    job = json.loads(first_job.read_text())
    print()
    print("Top-level keys (first coord):", list(job.keys()))
    print("session_name:               ", job.get("session_name"))
    print("recording_name:             ", job.get("recording_name"))

---
idx:                     0
min_recording_duration:  1.0
debug_mode:              False
coordinate_output_root:  ../../../obi-output/01_aind_ephys_dispatch/grid_scan/0
job files:               ['job_0.json']

Top-level keys (first coord): ['session_name', 'recording_name', 'recording_dict', 'skip_times', 'duration', 'input_folder', 'debug', 'multi_input']
session_name:                session0
recording_name:              block0_None_recording1
